# SQL queries

Reusable SQL and helpers against the project database (`project_config`: `DATA_DIR` / `DB_FILENAME`, usually `data/raw/events.db`). **Run the setup cell first.**

Table reference: [`docs/DB_SCHEMA.md`](../docs/DB_SCHEMA.md).

## `author_association` (GitHub)

The project reads the first non-empty of `payload.comment`, `payload.review`, `payload.pull_request`, or `payload.issue` (`author_association` lives on whichever object applies). Values match GitHub’s **`CommentAuthorAssociation`** enum (REST / webhooks / GraphQL)—see [GitHub docs](https://docs.github.com/en/graphql/reference/enums#commentauthorassociation).

| Value | Meaning (GitHub) |
|---|---|
| **`OWNER`** | Owner of the repository. |
| **`MEMBER`** | Member of the organization that owns the repository. |
| **`COLLABORATOR`** | Invited to collaborate on the repository. |
| **`CONTRIBUTOR`** | Has previously committed to the repository. |
| **`FIRST_TIMER`** | Has not previously committed on GitHub. |
| **`FIRST_TIME_CONTRIBUTOR`** | Has not previously committed to this repository. |
| **`NONE`** | No association with the repository. |
| **`MANNEQUIN`** | Placeholder for an unclaimed / imported user. |

Your dumps may omit some of these entirely. Empty or missing JSON yields `(empty)` in segmentation helpers.

## `type` (GitHub event)

Top-level **`$.type`** is the GitHub timeline / webhook event name (same strings as `json_extract(event_data, '$.type')`). Full detail: [`docs/DB_SCHEMA.md`](../docs/DB_SCHEMA.md) — *Meaning of top-level `type`*.

| Value | Meaning (short) |
|---|---|
| **`PullRequestEvent`** | PR lifecycle activity (open, close, edit, label, sync, …); payload focuses on **`pull_request`**. |
| **`PullRequestReviewEvent`** | A PR **review** submitted (approve / request changes / review-level comment); **`review`**. |
| **`PullRequestReviewCommentEvent`** | **Inline** comment on a **line** of the PR diff; **`comment`**. |
| **`IssueCommentEvent`** | **Thread** comment on an issue or PR (PRs are issues); **`comment`**. |

Other `type` strings are possible if you widen extraction. Missing/blank yields `(missing)` in segmentation helpers.

## Segmentations: **repo** × **author_association**, **repo** × **event type**, **event type**

Under each of `events` / `cleaned` / `scores`, the **Segmentations** code cell calls `run_breakdowns_for_scope(scope)`, which shows: (1) **`breakdown_repo_author_association`** — `owner/repo` × **author_association**; (2) **`breakdown_repo_event_type`** — `owner/repo` × **`$.type`**; (3) **`breakdown_event_type`** — marginal counts by **`$.type`**. There is **no** programming-language / “repo style” bucket.

| `scope` | Row set |
|---------|--------|
| `"events"` | All rows in `events` |
| `"cleaned"` | `cleaned` inner join `events` (preprocess survivors) |
| `"scores"` | `scores` join `cleaned` join `events` (one row per judge model per comment) |

The setup cell leaves those calls **commented out** by default; uncomment to run.


In [166]:
import json
import sys
from html import escape
from pathlib import Path

import pandas as pd
import sqlite3
from IPython.display import HTML, Markdown, display

# --- configurable sample for deep JSON inspection ---
# Set to a specific GitHub event id string, or None to use the first row in `events`.
SAMPLE_EVENT_ID: str | None = 34516188931

# How much text to show in SQL previews (table peek cells).
EVENT_DATA_PREVIEW_LEN = 200
CLEANED_TEXT_PREVIEW_LEN = 220
TOKENS_PREVIEW_LEN = 140
REASONING_PREVIEW_LEN = 160


def _repo_root_for_notebook() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "project_config.py").is_file():
            return p
    return here


_REPO = _repo_root_for_notebook()
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from project_config import DATA_DIR, DB_FILENAME, repo_root

DB_PATH = repo_root() / DATA_DIR / DB_FILENAME

def author_association_sql(alias: str = "e") -> str:
    """Same paths as `preprocessing.workflow._get_author_association` / `judge/storage.CLEANED_JOIN_FILTERS`."""
    return (
        f"COALESCE("
        f"json_extract({alias}.event_data, '$.payload.comment.author_association'), "
        f"json_extract({alias}.event_data, '$.payload.review.author_association'), "
        f"json_extract({alias}.event_data, '$.payload.pull_request.author_association'), "
        f"json_extract({alias}.event_data, '$.payload.issue.author_association')"
        f")"
    )



def repo_name_sql(alias: str = "e") -> str:
    return f"json_extract({alias}.event_data, '$.repo.name')"


def event_type_sql(alias: str = "e") -> str:
    """Top-level GitHub / GHArchive event type (e.g. ``IssueCommentEvent``)."""
    return f"json_extract({alias}.event_data, '$.type')"


def comment_author_login_sql(alias: str = "e") -> str:
    """GitHub login for comment/review author; falls back to ``actor.login``."""
    return (
        f"COALESCE("
        f"json_extract({alias}.event_data, '$.payload.comment.user.login'), "
        f"json_extract({alias}.event_data, '$.payload.review.user.login'), "
        f"json_extract({alias}.event_data, '$.actor.login')"
        f")"
    )


# SQL FROM clause + join pattern per logical table (always expose `events` as `e` for JSON paths).
_BREAKDOWN_FROM = {
    "events": "FROM events e",
    "cleaned": "FROM cleaned c INNER JOIN events e ON e.id = c.id",
    "scores": (
        "FROM scores s "
        "INNER JOIN cleaned c ON c.id = s.comment_id "
        "INNER JOIN events e ON e.id = c.id"
    ),
}


def breakdown_df(
    scope: str,
    select_exprs: list[tuple[str, str]],
    group_by_positions: str,
    order_by: str = "n DESC",
) -> pd.DataFrame:
    """Run a grouped COUNT(*) query for scope in {'events','cleaned','scores'}.

    Use SQLite ``GROUP BY 1`` / ``GROUP BY 1, 2`` so grouping matches SELECT aliases.
    """
    if scope not in _BREAKDOWN_FROM:
        raise ValueError(f"scope must be one of {list(_BREAKDOWN_FROM)}")
    cols = ", ".join(f"{expr} AS {alias}" for expr, alias in select_exprs)
    q = f"""
    SELECT {cols}, COUNT(*) AS n
    {_BREAKDOWN_FROM[scope]}
    GROUP BY {group_by_positions}
    ORDER BY {order_by}
    """
    with connect() as conn:
        return pd.read_sql_query(q, conn)


def breakdown_repo_author_association(scope: str = "cleaned") -> pd.DataFrame:
    """Counts by full `owner/repo` (`$.repo.name`) × author_association."""
    rn = repo_name_sql("e")
    aa = author_association_sql("e")
    aa_label = f"COALESCE(NULLIF(TRIM({aa}), ''), '(empty)')"
    return breakdown_df(
        scope,
        [(rn, "repo"), (aa_label, "author_association")],
        "1, 2",
        order_by="repo ASC, author_association ASC, n DESC",
    )


def breakdown_event_type(scope: str = "cleaned") -> pd.DataFrame:
    """Counts by GH event ``type`` (``json_extract(..., '$.type')``)."""
    et = event_type_sql("e")
    type_label = f"COALESCE(NULLIF(TRIM({et}), ''), '(missing)')"
    return breakdown_df(
        scope,
        [(type_label, "event_type")],
        "1",
        order_by="n DESC, event_type ASC",
    )


def breakdown_repo_event_type(scope: str = "cleaned") -> pd.DataFrame:
    """Counts by full `owner/repo` (`$.repo.name`) × GH event `type` (`$.type`)."""
    rn = repo_name_sql("e")
    et = event_type_sql("e")
    repo_col = f"COALESCE(NULLIF(TRIM({rn}), ''), '(missing)')"
    type_label = f"COALESCE(NULLIF(TRIM({et}), ''), '(missing)')"
    return breakdown_df(
        scope,
        [(repo_col, "repo"), (type_label, "event_type")],
        "1, 2",
        order_by="repo ASC, event_type ASC, n DESC",
    )


pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", None)


def connect() -> sqlite3.Connection:
    return sqlite3.connect(str(DB_PATH))


def show_sample_event() -> None:
    """Pretty-print one row's `event_data` JSON in a scrollable panel."""
    with connect() as conn:
        if SAMPLE_EVENT_ID:
            row = conn.execute(
                "SELECT id, event_data FROM events WHERE id = ? LIMIT 1",
                (SAMPLE_EVENT_ID,),
            ).fetchone()
        else:
            row = conn.execute("SELECT id, event_data FROM events LIMIT 1").fetchone()
    if not row:
        display(Markdown("_No rows in `events`._"))
        return
    eid, raw = row
    try:
        pretty = json.dumps(json.loads(raw), indent=2, ensure_ascii=False)
    except json.JSONDecodeError:
        pretty = raw[:8000] + ("\n… [truncated]" if len(raw) > 8000 else "")
    display(Markdown(f"##### One sample `event_data` (`id={eid}`)"))
    safe = escape(pretty)
    html = (
        "<div style=\"max-height:28rem;overflow:auto;border:1px solid #ccc;"
        "padding:0.6rem;background:#f9f9f9;font-family:ui-monospace,monospace\">"
        f"<pre style=\"margin:0;white-space:pre-wrap;font-size:12px;line-height:1.35\">{safe}</pre>"
        "</div>"
    )
    display(HTML(html))


def peek_table(name: str, *, display_top_n: int = 2) -> None:
    """Row count (plus per-repo for `events` / `cleaned` / `scores`), PRAGMA layout, samples."""
    n = max(1, min(int(display_top_n), 50_000))
    with connect() as conn:
        display(Markdown("##### Row count"))
        display(pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {name}", conn))
        if name in ("events", "cleaned", "scores"):
            display(Markdown("##### Row count by repo (`$.repo.name`)"))
            rn = repo_name_sql("e")
            repo_col = f"COALESCE(NULLIF(TRIM({rn}), ''), '(missing)')"
            if name == "events":
                q_repo = f"""
                SELECT {repo_col} AS repo, COUNT(*) AS n
                FROM events e
                GROUP BY 1
                ORDER BY n DESC, repo ASC
                """
            elif name == "cleaned":
                q_repo = f"""
                SELECT {repo_col} AS repo, COUNT(*) AS n
                FROM cleaned c INNER JOIN events e ON e.id = c.id
                GROUP BY 1
                ORDER BY n DESC, repo ASC
                """
            else:
                q_repo = f"""
                SELECT {repo_col} AS repo, COUNT(*) AS n
                FROM scores s
                INNER JOIN cleaned c ON c.id = s.comment_id
                INNER JOIN events e ON e.id = c.id
                GROUP BY 1
                ORDER BY n DESC, repo ASC
                """
            display(pd.read_sql_query(q_repo, conn))
        display(Markdown("##### Columns"))
        display(pd.read_sql_query(f"PRAGMA table_info({name})", conn))
        display(Markdown(f"##### Sample ({n} rows, truncated previews)"))
        if name == "events":
            q = (
                f"SELECT id, substr(event_data, 1, {EVENT_DATA_PREVIEW_LEN}) || '…' AS event_data_preview "
                f"FROM events LIMIT {n}"
            )
        elif name == "cleaned":
            q = (
                "SELECT id, "
                f"substr(cleaned_text, 1, {CLEANED_TEXT_PREVIEW_LEN}) || '…' AS cleaned_text_preview, "
                f"substr(tokens, 1, {TOKENS_PREVIEW_LEN}) || '…' AS tokens_preview "
                f"FROM cleaned LIMIT {n}"
            )
        elif name == "scores":
            q = (
                "SELECT comment_id, model_name, "
                "fun_score, nsi_score, insi_score, isi_score, "
                f"substr(fun_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS fun_reasoning_preview, "
                f"substr(nsi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS nsi_reasoning_preview, "
                f"substr(insi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS insi_reasoning_preview, "
                f"substr(isi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS isi_reasoning_preview "
                f"FROM scores LIMIT {n}"
            )
        else:
            q = f"SELECT * FROM {name} LIMIT {n}"
        display(pd.read_sql_query(q, conn))




def run_breakdowns_for_scope(scope: str) -> None:
    """For one scope: repo × aa, repo × event type, then marginal counts by ``$.type``."""
    with connect() as conn:
        tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if scope == "scores" and "scores" not in tables:
        display(Markdown(f"*Scope `{scope}`: table `scores` missing — run the judge.*"))
        return
    if scope == "cleaned" and "cleaned" not in tables:
        display(Markdown(f"*Scope `{scope}`: table `cleaned` missing.*"))
        return
    prev_max = pd.get_option("display.max_rows")
    try:
        pd.set_option("display.max_rows", 150)
        display(Markdown("**Repo × author_association** — `$.repo.name` (full repo, not a language bucket)"))
        display(breakdown_repo_author_association(scope))
        display(Markdown("**Repo × event type** — `$.repo.name` × top-level `$.type`"))
        display(breakdown_repo_event_type(scope))
        display(Markdown("**Event type** — marginal counts on top-level `$.type`"))
        display(breakdown_event_type(scope))
    finally:
        pd.set_option("display.max_rows", prev_max)


def run_all_breakdowns(
    scopes: tuple[str, ...] = ("events", "cleaned", "scores"),
) -> None:
    """Optional: run `run_breakdowns_for_scope` for every table scope."""
    for scope in scopes:
        run_breakdowns_for_scope(scope)


SCORE_COLUMNS: tuple[str, ...] = ("fun_score", "nsi_score", "insi_score", "isi_score")


def load_scores_with_metadata() -> pd.DataFrame:
    """``scores`` joined to ``events`` (via ``cleaned``). One row per (``comment_id``, ``model_name``)."""
    rn = repo_name_sql("e")
    aa = author_association_sql("e")
    ul = comment_author_login_sql("e")
    et = event_type_sql("e")
    repo_col = f"COALESCE(NULLIF(TRIM({rn}), ''), '(missing)')"
    aa_label = f"COALESCE(NULLIF(TRIM({aa}), ''), '(empty)')"
    user_col = f"COALESCE(NULLIF(TRIM({ul}), ''), '(missing)')"
    type_col = f"COALESCE(NULLIF(TRIM({et}), ''), '(missing)')"
    q = f"""
    SELECT
      s.comment_id,
      s.model_name,
      s.fun_score,
      s.nsi_score,
      s.insi_score,
      s.isi_score,
      {repo_col} AS repo,
      {aa_label} AS author_association,
      {user_col} AS user_login,
      {type_col} AS event_type
    FROM scores s
    INNER JOIN cleaned c ON c.id = s.comment_id
    INNER JOIN events e ON e.id = c.id
    """
    with connect() as conn:
        return pd.read_sql_query(q, conn)


def score_stats_grouped(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Per group: ``n_rows``, per score column ``_n`` and ``_avg``/``_mean``/``_median``/``_p25`` … ``_p99``."""
    rows: list[dict[str, object]] = []
    for keys, g in df.groupby(group_cols, dropna=False):
        keys_t = keys if isinstance(keys, tuple) else (keys,)
        row = {group_cols[i]: keys_t[i] for i in range(len(group_cols))}
        row["n_rows"] = len(g)
        for c in SCORE_COLUMNS:
            s = g[c].dropna()
            row[f"{c}_n"] = int(s.shape[0])
            if s.empty:
                nan = float("nan")
                for suffix in (
                    "avg",
                    "mean",
                    "median",
                    "p25",
                    "p50",
                    "p75",
                    "p99",
                ):
                    row[f"{c}_{suffix}"] = nan
            else:
                mean = float(s.mean())
                row[f"{c}_avg"] = mean
                row[f"{c}_mean"] = mean
                row[f"{c}_median"] = float(s.median())
                row[f"{c}_p25"] = float(s.quantile(0.25))
                row[f"{c}_p50"] = float(s.quantile(0.50))
                row[f"{c}_p75"] = float(s.quantile(0.75))
                row[f"{c}_p99"] = float(s.quantile(0.99))
        rows.append(row)
    out = pd.DataFrame(rows)
    return out.sort_values(group_cols, kind="mergesort").reset_index(drop=True)


def score_stats_by_repo() -> pd.DataFrame:
    """FUN/NSI/INSI/ISI summaries grouped by ``repo`` only."""
    return score_stats_grouped(load_scores_with_metadata(), ["repo"])


def score_stats_by_repo_author_association() -> pd.DataFrame:
    """Same summaries grouped by ``repo`` × ``author_association``."""
    return score_stats_grouped(load_scores_with_metadata(), ["repo", "author_association"])


def score_stats_by_repo_event_type() -> pd.DataFrame:
    """Summaries grouped by ``repo`` × ``event_type`` (top-level ``$.type``)."""
    return score_stats_grouped(load_scores_with_metadata(), ["repo", "event_type"])


def score_stats_by_repo_aa_user() -> pd.DataFrame:
    """Summaries grouped by ``repo`` × ``author_association`` × ``user_login`` (comment/review author)."""
    return score_stats_grouped(
        load_scores_with_metadata(),
        ["repo", "author_association", "user_login"],
    )


def score_stats_by_repo_aa_user_event_type() -> pd.DataFrame:
    """Summaries grouped by ``repo`` × ``author_association`` × ``user_login`` × GH event ``$.type``."""
    return score_stats_grouped(
        load_scores_with_metadata(),
        ["repo", "author_association", "user_login", "event_type"],
    )


def show_score_stats(which: str = "both") -> None:
    """Print wide tables. ``which``: ``repo`` | ``repo_aa`` | ``repo_type`` | ``repo_aa_user`` | ``repo_aa_user_type`` | ``both`` | ``all``."""
    with connect() as conn:
        tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if "scores" not in tables:
        display(Markdown("_No ``scores`` table — run the judge._"))
        return
    if "cleaned" not in tables:
        display(Markdown("_No ``cleaned`` table._"))
        return
    prev_cols = pd.get_option("display.max_columns")
    prev_width = pd.get_option("display.width")
    try:
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", 240)
        prev_rows = pd.get_option("display.max_rows")
        if which in ("repo", "both", "all"):
            display(
                Markdown(
                    "##### Score stats by **repo** — avg/mean = arithmetic average; "
                    "p50 = sample median (pandas ``median`` / ``quantile(0.5)``)."
                )
            )
            display(score_stats_by_repo())
        if which in ("repo_aa", "both", "all"):
            display(Markdown("##### Score stats by **repo × author_association**"))
            display(score_stats_by_repo_author_association())
        if which in ("repo_type", "all"):
            display(
                Markdown(
                    "##### Score stats by **repo × event type** — top-level ``$.type`` "
                    "(e.g. ``IssueCommentEvent``, ``PullRequestReviewCommentEvent``)"
                )
            )
            pd.set_option("display.max_rows", 500)
            display(score_stats_by_repo_event_type())
            pd.set_option("display.max_rows", prev_rows)
        if which in ("repo_aa_user", "all"):
            display(
                Markdown(
                    "##### Score stats by **repo × author_association × user_login** "
                    "(``payload.comment.user.login`` / ``review.user.login``, else ``actor.login``)"
                )
            )
            pd.set_option("display.max_rows", 500)
            display(score_stats_by_repo_aa_user())
            pd.set_option("display.max_rows", prev_rows)
        if which in ("repo_aa_user_type", "all"):
            display(
                Markdown(
                    "##### Score stats by **repo × author_association × user_login × event type** "
                    "(top-level ``$.type`` on ``events.event_data``)"
                )
            )
            pd.set_option("display.max_rows", 800)
            display(score_stats_by_repo_aa_user_event_type())
            pd.set_option("display.max_rows", prev_rows)
    finally:
        pd.set_option("display.max_columns", prev_cols)
        pd.set_option("display.width", prev_width)


print(f"DB: {DB_PATH.resolve()}  exists={DB_PATH.is_file()}")


DB: /Users/naataaniitsosie/repos/swe-principals/data/raw/events.db  exists=True


## One sample `event_data`

Runs `show_sample_event()` using `SAMPLE_EVENT_ID` from the setup cell (`None` = first row in `events`).


In [167]:
# show_sample_event()


## `events`

Peek uses truncated `event_data` previews so the notebook stays small. For one full JSON object, run the **One sample `event_data`** cell above.

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM events;` |
| Layout | `PRAGMA table_info(events);` |
| Sample (short) | `… FROM events LIMIT N` — `peek_table("events", display_top_n=2)` (default **2**). |

**Segmentations:** Uncomment the **Segmentations** cells directly under `peek_table("events")`. `event_data` has top-level `$.type`, `$.repo.name`, and `author_association` on payload objects. Run **`run_breakdowns_for_scope("events")`** (repo × `author_association`, repo × `$.type`, marginal `$.type`).



In [168]:
peek_table("events")


##### Row count

,n
0,41936


##### Row count by repo (`$.repo.name`)

,repo,n
0,django/django,13904
1,spring-projects/spring-boot,8788
2,nestjs/nest,4843
3,tiangolo/fastapi,4486
4,fastify/fastify,4479
5,expressjs/express,3245
6,gin-gonic/gin,1263
7,pallets/flask,505
8,koajs/koa,241
9,hapijs/hapi,182


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,1
1,1,event_data,TEXT,1,None,0


##### Sample (2 rows, truncated previews)

,id,event_data_preview
0,34509224090,"{""id"": ""34509224090"", ""type"": ""PullRequestEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_url…"
1,34509271007,"{""id"": ""34509271007"", ""type"": ""IssueCommentEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_ur…"


### Segmentations (`scope="events"`)

Runs **`run_breakdowns_for_scope`** from the setup cell: **repo × author_association**, **repo × event type**, then marginal **event type**. **Commented out by default.**


In [169]:
run_breakdowns_for_scope("events")


**Repo × author_association** — `$.repo.name` (full repo, not a language bucket)

,repo,author_association,n
0,django/django,CONTRIBUTOR,6983
1,django/django,MEMBER,4303
2,django/django,NONE,2618
3,expressjs/express,CONTRIBUTOR,124
4,expressjs/express,MEMBER,1353
5,expressjs/express,NONE,1768
6,fastify/fastify,CONTRIBUTOR,1026
7,fastify/fastify,MEMBER,2211
8,fastify/fastify,NONE,1242
9,gin-gonic/gin,CONTRIBUTOR,307


**Repo × event type** — `$.repo.name` × top-level `$.type`

,repo,event_type,n
0,django/django,IssueCommentEvent,3405
1,django/django,PullRequestEvent,2368
2,django/django,PullRequestReviewCommentEvent,5269
3,django/django,PullRequestReviewEvent,2862
4,expressjs/express,IssueCommentEvent,1526
5,expressjs/express,PullRequestEvent,1165
6,expressjs/express,PullRequestReviewCommentEvent,292
7,expressjs/express,PullRequestReviewEvent,262
8,fastify/fastify,IssueCommentEvent,2084
9,fastify/fastify,PullRequestEvent,790


**Event type** — marginal counts on top-level `$.type`

,event_type,n
0,IssueCommentEvent,20324
1,PullRequestEvent,9165
2,PullRequestReviewCommentEvent,7968
3,PullRequestReviewEvent,4479


## `cleaned`

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM cleaned;` |
| Columns | `PRAGMA table_info(cleaned);` |
| Sample | Truncated columns; `LIMIT N` via `peek_table("cleaned", display_top_n=2)` (default **2**). |

**Segmentations:** Uncomment the segmentations cells under this section. Join `cleaned` → `events` on `id` to read JSON. Run **`run_breakdowns_for_scope("cleaned")`** for **repo × author_association**, **repo × event type**, and marginal **`$.type`** on preprocess survivors only.


In [170]:
peek_table("cleaned")


##### Row count

,n
0,37197


##### Row count by repo (`$.repo.name`)

,repo,n
0,django/django,12565
1,spring-projects/spring-boot,8464
2,tiangolo/fastapi,4243
3,nestjs/nest,3649
4,fastify/fastify,3375
5,expressjs/express,3059
6,gin-gonic/gin,1021
7,pallets/flask,450
8,koajs/koa,191
9,hapijs/hapi,180


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,1
1,1,cleaned_text,TEXT,1,None,0
2,2,tokens,TEXT,1,None,0


##### Sample (2 rows, truncated previews)

,id,cleaned_text_preview,tokens_preview
0,34509224090,fixed #35072 -- corrected field.choices description in models topic. fixed choices description in docs: [ticket](https://code.djangoproject.com/ticket/35072)…,"[""fixed"", ""35072"", ""corrected"", ""field"", ""choices"", ""description"", ""in"", ""models"", ""topic"", ""fixed"", ""choices"", ""description"", ""in"", ""docs"",…"
1,34509271007,"@expert-m thanks for this patch :+1: welcome aboard :boat: > @felixxm please take a look. looks good, thanks :+1:…","[""expert"", ""m"", ""thanks"", ""for"", ""this"", ""patch"", ""1"", ""welcome"", ""aboard"", ""boat"", ""felixxm"", ""please"", ""take"", ""a"", ""look"", ""looks"", ""good…"


### Segmentations (`scope="cleaned"`)

Runs **`run_breakdowns_for_scope`** from the setup cell: **repo × author_association**, **repo × event type**, then marginal **event type**. **Commented out by default.**


In [171]:
run_breakdowns_for_scope("cleaned")


**Repo × author_association** — `$.repo.name` (full repo, not a language bucket)

,repo,author_association,n
0,django/django,CONTRIBUTOR,6404
1,django/django,MEMBER,4061
2,django/django,NONE,2100
3,expressjs/express,CONTRIBUTOR,111
4,expressjs/express,MEMBER,1248
5,expressjs/express,NONE,1700
6,fastify/fastify,CONTRIBUTOR,807
7,fastify/fastify,MEMBER,1834
8,fastify/fastify,NONE,734
9,gin-gonic/gin,CONTRIBUTOR,186


**Repo × event type** — `$.repo.name` × top-level `$.type`

,repo,event_type,n
0,django/django,IssueCommentEvent,3334
1,django/django,PullRequestEvent,2358
2,django/django,PullRequestReviewCommentEvent,5092
3,django/django,PullRequestReviewEvent,1781
4,expressjs/express,IssueCommentEvent,1492
5,expressjs/express,PullRequestEvent,1146
6,expressjs/express,PullRequestReviewCommentEvent,278
7,expressjs/express,PullRequestReviewEvent,143
8,fastify/fastify,IssueCommentEvent,1602
9,fastify/fastify,PullRequestEvent,683


**Event type** — marginal counts on top-level `$.type`

,event_type,n
0,IssueCommentEvent,19030
1,PullRequestEvent,7921
2,PullRequestReviewCommentEvent,7620
3,PullRequestReviewEvent,2626


## `scores`

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM scores;` |
| Columns | `PRAGMA table_info(scores);` |
| Sample | Truncated reasoning; `LIMIT N` via `peek_table("scores", display_top_n=2)` (default **2**). |

**Segmentations:** Uncomment the segmentations cells under this section. Join `scores` → `cleaned` → `events`. Run **`run_breakdowns_for_scope("scores")`** for **repo × author_association**, **repo × event type**, and marginal **`$.type`**; each `scores` row is one judge run, so counts can exceed unique comments.

**Score distributions:** At the bottom, **`show_score_stats()`** prints score stats by **repo**, **repo × author_association**, **`show_score_stats("repo_type")`** (**repo × event `$.type`**), **`show_score_stats("repo_aa_user")`**, **`show_score_stats("repo_aa_user_type")`**, or **`show_score_stats("all")`**.


In [172]:
peek_table("scores", display_top_n=2)


##### Row count

,n
0,100


##### Row count by repo (`$.repo.name`)

,repo,n
0,expressjs/express,100


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,comment_id,TEXT,1,None,1
1,1,model_name,TEXT,1,None,2
2,2,fun_score,INTEGER,1,None,0
3,3,fun_reasoning,TEXT,1,None,0
4,4,nsi_score,INTEGER,1,None,0
5,5,nsi_reasoning,TEXT,1,None,0
6,6,insi_score,INTEGER,1,None,0
7,7,insi_reasoning,TEXT,1,None,0
8,8,isi_score,INTEGER,1,None,0
9,9,isi_reasoning,TEXT,1,None,0


##### Sample (2 rows, truncated previews)

,comment_id,model_name,fun_score,nsi_score,insi_score,isi_score,fun_reasoning_preview,nsi_reasoning_preview,insi_reasoning_preview,isi_reasoning_preview
0,34517387521,gpt-5.4-mini,0,0,0,0,The comment only requests updating a README file and does not indicate any runtime defect or broken behavior.…,It does not explicitly invoke team conventions or a group rule about documentation updates; it is just a terse request.…,"There is no implicit social pressure conveyed beyond the bare request, and no unstated norm can be inferred from the text alone.…","No expert rationale, documentation standard, or technical justification is provided.…"
1,34548195471,gpt-5.4-mini,0,0,0,0,The comment is only a request to update documentation; it does not indicate a runtime or correctness issue in the code.…,"There is no explicit appeal to team norms, project conventions, or belonging. It simply asks for a README update.…",The request is direct but gives no reasoning that would imply an unstated social rule or hidden expectation beyond the literal task.…,"No technical justification, documentation reference, or expert rationale is provided—just a bare instruction to update the README.…"


### Segmentations (`scope="scores"`)

Runs **`run_breakdowns_for_scope`** from the setup cell: **repo × author_association**, **repo × event type**, then marginal **event type**. **Commented out by default.**


In [173]:
run_breakdowns_for_scope("scores")


**Repo × author_association** — `$.repo.name` (full repo, not a language bucket)

,repo,author_association,n
0,expressjs/express,MEMBER,8
1,expressjs/express,NONE,92


**Repo × event type** — `$.repo.name` × top-level `$.type`

,repo,event_type,n
0,expressjs/express,IssueCommentEvent,38
1,expressjs/express,PullRequestEvent,58
2,expressjs/express,PullRequestReviewCommentEvent,1
3,expressjs/express,PullRequestReviewEvent,3


**Event type** — marginal counts on top-level `$.type`

,event_type,n
0,PullRequestEvent,58
1,IssueCommentEvent,38
2,PullRequestReviewEvent,3
3,PullRequestReviewCommentEvent,1


### Score distributions (FUN / NSI / INSI / ISI)

Joins `scores` → `cleaned` → `events`. Each **scores row** is one judge run (`model_name`), so statistics are over judge rows, not necessarily unique comments.

Helpers include `score_stats_by_repo_event_type()` (**repo × `$.type`**), `score_stats_by_repo_aa_user_event_type()`, etc. **`show_score_stats("repo_type")`** / **`show_score_stats("all")`**. Same columns per score (`avg`, `mean`, `median`, `p25`–`p99`).


In [174]:
# Uncomment only **one** `show_score_stats(...)`: `both` = repo + repo_aa; `repo_type` = repo × $.type; `repo_aa_user_type` = 4-way; `all` runs repo, repo_aa, repo_type, repo_aa_user, repo_aa_user_type.
# show_score_stats()
# show_score_stats("repo")
# show_score_stats("repo_aa")  # aa = author_association
# show_score_stats("repo_aa_user")
# show_score_stats("repo_type")  # repo × event $.type
# show_score_stats("repo_aa_user_type")  # repo × aa × user_login × $.type
show_score_stats("all")


##### Score stats by **repo** — avg/mean = arithmetic average; p50 = sample median (pandas ``median`` / ``quantile(0.5)``).

,repo,n_rows,fun_score_n,fun_score_avg,fun_score_mean,fun_score_median,fun_score_p25,fun_score_p50,fun_score_p75,fun_score_p99,nsi_score_n,nsi_score_avg,nsi_score_mean,nsi_score_median,nsi_score_p25,nsi_score_p50,nsi_score_p75,nsi_score_p99,insi_score_n,insi_score_avg,insi_score_mean,insi_score_median,insi_score_p25,insi_score_p50,insi_score_p75,insi_score_p99,isi_score_n,isi_score_avg,isi_score_mean,isi_score_median,isi_score_p25,isi_score_p50,isi_score_p75,isi_score_p99
0,expressjs/express,100,100,0.38,0.38,0.0,0.0,0.0,0.0,3.0,100,0.06,0.06,0.0,0.0,0.0,0.0,3.0,100,0.21,0.21,0.0,0.0,0.0,0.0,2.0,100,0.46,0.46,0.0,0.0,0.0,1.0,3.0


##### Score stats by **repo × author_association**

,repo,author_association,n_rows,fun_score_n,fun_score_avg,fun_score_mean,fun_score_median,fun_score_p25,fun_score_p50,fun_score_p75,fun_score_p99,nsi_score_n,nsi_score_avg,nsi_score_mean,nsi_score_median,nsi_score_p25,nsi_score_p50,nsi_score_p75,nsi_score_p99,insi_score_n,insi_score_avg,insi_score_mean,insi_score_median,insi_score_p25,insi_score_p50,insi_score_p75,insi_score_p99,isi_score_n,isi_score_avg,isi_score_mean,isi_score_median,isi_score_p25,isi_score_p50,isi_score_p75,isi_score_p99
0,expressjs/express,MEMBER,8,8,0.375000,0.375000,0.0,0.0,0.0,0.25,1.93,8,0.75,0.75,0.0,0.0,0.0,0.75,3.0,8,0.500000,0.500000,0.0,0.0,0.0,1.0,1.93,8,0.750000,0.750000,0.5,0.0,0.5,1.25,2.0
1,expressjs/express,NONE,92,92,0.380435,0.380435,0.0,0.0,0.0,0.00,3.00,92,0.00,0.00,0.0,0.0,0.0,0.00,0.0,92,0.184783,0.184783,0.0,0.0,0.0,0.0,2.00,92,0.434783,0.434783,0.0,0.0,0.0,1.00,3.0


##### Score stats by **repo × event type** — top-level ``$.type`` (e.g. ``IssueCommentEvent``, ``PullRequestReviewCommentEvent``)

,repo,event_type,n_rows,fun_score_n,fun_score_avg,fun_score_mean,fun_score_median,fun_score_p25,fun_score_p50,fun_score_p75,fun_score_p99,nsi_score_n,nsi_score_avg,nsi_score_mean,nsi_score_median,nsi_score_p25,nsi_score_p50,nsi_score_p75,nsi_score_p99,insi_score_n,insi_score_avg,insi_score_mean,insi_score_median,insi_score_p25,insi_score_p50,insi_score_p75,insi_score_p99,isi_score_n,isi_score_avg,isi_score_mean,isi_score_median,isi_score_p25,isi_score_p50,isi_score_p75,isi_score_p99
0,expressjs/express,IssueCommentEvent,38,38,0.789474,0.789474,0.0,0.0,0.0,2.0,3.0,38,0.157895,0.157895,0.0,0.0,0.0,0.0,3.0,38,0.342105,0.342105,0.0,0.0,0.0,0.0,2.0,38,1.078947,1.078947,1.0,0.25,1.0,2.0,3.00
1,expressjs/express,PullRequestEvent,58,58,0.103448,0.103448,0.0,0.0,0.0,0.0,3.0,58,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,58,0.137931,0.137931,0.0,0.0,0.0,0.0,2.0,58,0.034483,0.034483,0.0,0.00,0.0,0.0,1.00
2,expressjs/express,PullRequestReviewCommentEvent,1,1,2.000000,2.000000,2.0,2.0,2.0,2.0,2.0,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1,2.000000,2.000000,2.0,2.00,2.0,2.0,2.00
3,expressjs/express,PullRequestReviewEvent,3,3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,3,0.333333,0.333333,0.0,0.00,0.0,0.5,0.98


##### Score stats by **repo × author_association × user_login** (``payload.comment.user.login`` / ``review.user.login``, else ``actor.login``)

,repo,author_association,user_login,n_rows,fun_score_n,fun_score_avg,fun_score_mean,fun_score_median,fun_score_p25,fun_score_p50,fun_score_p75,fun_score_p99,nsi_score_n,nsi_score_avg,nsi_score_mean,nsi_score_median,nsi_score_p25,nsi_score_p50,nsi_score_p75,nsi_score_p99,insi_score_n,insi_score_avg,insi_score_mean,insi_score_median,insi_score_p25,insi_score_p50,insi_score_p75,insi_score_p99,isi_score_n,isi_score_avg,isi_score_mean,isi_score_median,isi_score_p25,isi_score_p50,isi_score_p75,isi_score_p99
0,expressjs/express,MEMBER,UlisesGascon,5,5,0.400000,0.400000,0.0,0.0,0.0,0.0,1.92,5,0.6,0.6,0.0,0.0,0.0,0.0,2.88,5,0.400000,0.400000,0.0,0.00,0.0,0.00,1.92,5,0.600000,0.600000,0.0,0.00,0.0,1.00,1.96
1,expressjs/express,MEMBER,dougwilson,3,3,0.333333,0.333333,0.0,0.0,0.0,0.5,0.98,3,1.0,1.0,0.0,0.0,0.0,1.5,2.94,3,0.666667,0.666667,1.0,0.50,1.0,1.00,1.00,3,1.000000,1.000000,1.0,0.50,1.0,1.50,1.98
2,expressjs/express,NONE,24ankit,2,2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
3,expressjs/express,NONE,821232,2,2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
4,expressjs/express,NONE,AONTU-ROY,1,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
5,expressjs/express,NONE,ATG6174,1,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
6,expressjs/express,NONE,Arifali1402,1,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,1.000000,1.000000,1.0,1.00,1.0,1.00,1.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
7,expressjs/express,NONE,Csiddharth7906,1,1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
8,expressjs/express,NONE,Dev1392003,3,3,1.000000,1.000000,0.0,0.0,0.0,1.5,2.94,3,0.0,0.0,0.0,0.0,0.0,0.0,0.00,3,0.333333,0.333333,0.0,0.00,0.0,0.50,0.98,3,0.666667,0.666667,0.0,0.00,0.0,1.00,1.96
9,expressjs/express,NONE,DmytroKondrashov,1,1,3.000000,3.000000,3.0,3.0,3.0,3.0,3.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00


##### Score stats by **repo × author_association × user_login × event type** (top-level ``$.type`` on ``events.event_data``)

,repo,author_association,user_login,event_type,n_rows,fun_score_n,fun_score_avg,fun_score_mean,fun_score_median,fun_score_p25,fun_score_p50,fun_score_p75,fun_score_p99,nsi_score_n,nsi_score_avg,nsi_score_mean,nsi_score_median,nsi_score_p25,nsi_score_p50,nsi_score_p75,nsi_score_p99,insi_score_n,insi_score_avg,insi_score_mean,insi_score_median,insi_score_p25,insi_score_p50,insi_score_p75,insi_score_p99,isi_score_n,isi_score_avg,isi_score_mean,isi_score_median,isi_score_p25,isi_score_p50,isi_score_p75,isi_score_p99
0,expressjs/express,MEMBER,UlisesGascon,IssueCommentEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,3.0,3.0,3.0,3.0,3.0,3.0,3.00,1,2.000000,2.000000,2.0,2.00,2.0,2.00,2.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
1,expressjs/express,MEMBER,UlisesGascon,PullRequestReviewCommentEvent,1,1,2.000000,2.000000,2.0,2.00,2.0,2.00,2.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,2.000000,2.000000,2.0,2.00,2.0,2.00,2.00
2,expressjs/express,MEMBER,UlisesGascon,PullRequestReviewEvent,3,3,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,3,0.0,0.0,0.0,0.0,0.0,0.0,0.00,3,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,3,0.333333,0.333333,0.0,0.00,0.0,0.50,0.98
3,expressjs/express,MEMBER,dougwilson,IssueCommentEvent,3,3,0.333333,0.333333,0.0,0.00,0.0,0.50,0.98,3,1.0,1.0,0.0,0.0,0.0,1.5,2.94,3,0.666667,0.666667,1.0,0.50,1.0,1.00,1.00,3,1.000000,1.000000,1.0,0.50,1.0,1.50,1.98
4,expressjs/express,NONE,24ankit,IssueCommentEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
5,expressjs/express,NONE,24ankit,PullRequestEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
6,expressjs/express,NONE,821232,PullRequestEvent,2,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,2,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
7,expressjs/express,NONE,AONTU-ROY,PullRequestEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
8,expressjs/express,NONE,ATG6174,PullRequestEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
9,expressjs/express,NONE,Arifali1402,PullRequestEvent,1,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00,1,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1,1.000000,1.000000,1.0,1.00,1.0,1.00,1.00,1,0.000000,0.000000,0.0,0.00,0.0,0.00,0.00
